In [ ]:
import h5py
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

# Class-weighted wrapper matching pipeline logic
class ClassWeightedXGB(XGBClassifier):
    def __init__(self, class_weight=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weight = class_weight

    def fit(self, X, y, **kwargs):
        if self.class_weight is not None:
            sample_weight = compute_sample_weight(class_weight=self.class_weight, y=y)
            return super().fit(X, y, sample_weight=sample_weight, **kwargs)
        return super().fit(X, y, **kwargs)

# --- 1. LOAD EMBEDDINGS FROM HDF5 ---
H5_PATH = "../embeddings/task4_dna_rna_Vir2vec-422M.h5"

with h5py.File(H5_PATH, "r") as f:
    X = np.array(f["embeddings"][:])
    raw_labels = [l.decode("utf-8") if isinstance(l, bytes) else str(l) for l in f["labels"][:]]

y = LabelEncoder().fit_transform(raw_labels)

# --- 2. PIPELINE & HYPERPARAMETER GRID ---
pipeline = Pipeline([
    ("clf", ClassWeightedXGB(
        n_estimators=50,
        tree_method="hist",
        max_bin=128,
        subsample=0.8,
        colsample_bytree=0.25,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    ))
])

param_grid = {
    "clf__class_weight": [None, "balanced"]
}

# --- 3. NESTED 5x3 STRATIFIED CROSS-VALIDATION ---
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42 + fold_idx)
    grid = GridSearchCV(pipeline, param_grid, scoring="balanced_accuracy", cv=inner_cv, n_jobs=-1)
    grid.fit(X_train, y_train)
    
    y_pred = grid.best_estimator_.predict(X_test)
    
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    fold_metrics.append({
        "fold": fold_idx,
        "balanced_accuracy": bal_acc,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred)
    })
    print(f"Fold {fold_idx}: Balanced Acc = {bal_acc:.4f}")

# --- 4. SUMMARY ---
df_res = pd.DataFrame(fold_metrics)
print("\n--- Final XGBoost Results ---")
print(f"Macro Balanced Accuracy: {df_res['balanced_accuracy'].mean():.4f} ± {df_res['balanced_accuracy'].std():.4f}")